# Zuzai work-type labels — fine-tuning on Colab (resumable)

**How to run:** Runtime → Change runtime type → GPU (A100 best, L4 fine; T4 works for base models only) → **Runtime → Run all**.

- One job = (model, fold). Checkpoints are saved to Google Drive (`My Drive/zuzai-hw/ft/…/ckpt`) every 300 steps.
  If the GPU is taken away, reconnect and **Run all** again: finished folds are skipped and the current one resumes from its last checkpoint.
- After every finished fold, `My Drive/zuzai-hw/exports/ft_<model>.zip` is refreshed with all finished folds of that model
  (validation probabilities, holdout probabilities, metrics in `done.json`). Download them and on the Mac run
  `uv run python src/import_colab.py ~/Downloads/ft_*.zip`
- Logs: `My Drive/zuzai-hw/logs/ft_<model>.log` (per-fold val macro-F1 / min-F1 / time).

In [ ]:
# ---- Config (edit only if needed) ----
# (model key from src/finetune.py, folds). Fold 0 of each candidate first (model selection), then the other folds.
JOBS = [
    ("deberta-v3-base", [0]),
    ("modernbert-large", [0]),
    ("qwen3-0.6b-lora", [0]),
    ("deberta-v3-base", [1, 2, 3, 4]),
]
EPOCHS = 3
CLASS_WEIGHT = "inv"     # inv (as Zuzai's DistilBERT) | sqrt_inv | none
REPO = "https://github.com/ShayHummel/work-type-labels.git"
BRANCH = "solution"
OUT = "/content/drive/MyDrive/zuzai-hw"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
os.makedirs(OUT, exist_ok=True)
if not os.path.exists("/content/work-type-labels"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/work-type-labels"], check=True)
else:
    subprocess.run(["git", "-C", "/content/work-type-labels", "pull", "-q"], check=True)
%cd /content/work-type-labels
!git log --oneline -1

In [ ]:
!pip install -q "transformers==5.18.0" "accelerate==1.15.0" "peft==0.21.2" "sentencepiece==0.2.2" "protobuf"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
for model, folds in JOBS:
    print(f"===== {model} folds {folds} =====", flush=True)
    !python src/finetune.py --model {model} --folds {" ".join(map(str, folds))} --out {OUT} --epochs {EPOCHS} --class-weight {CLASS_WEIGHT}

In [ ]:
!ls -lh {OUT}/exports
!grep -h DONE {OUT}/logs/ft_*.log